# DendryNexus Universal Decompiler

This notebook allows you to decompile a `core.js` file from a DendryNexus project back into its original `.dry` source files with 1:1 fidelity (if the `core.js` was built with source preservation metadata).

In [ ]:
import os
import json
import shutil
import requests
from google.colab import files

def decompile(core_js_path, output_dir='decompiled_source'):
    print(f"Processing {core_js_path}...")
    with open(core_js_path, 'r', encoding='utf-8') as f:
        content = f.read()

    prefix = "window.game="
    start_idx = content.find(prefix)
    if start_idx == -1:
        print("Error: Could not find window.game in file.")
        return

    json_part = content[start_idx + len(prefix):]
    try:
        decoder = json.JSONDecoder()
        game_wrapper, end = decoder.raw_decode(json_part)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return

    compiled_json_str = game_wrapper.get('compiled')
    if not compiled_json_str:
        print("Error: Compiled game data not found.")
        return

    game = json.loads(compiled_json_str)

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    files_to_write = {}
    def collect_files(obj):
        if isinstance(obj, dict):
            if '$metadata' in obj:
                metadata = obj['$metadata']
                if '$raw' in metadata and '$file' in metadata:
                    files_to_write[metadata['$file']] = metadata['$raw']
            for val in obj.values():
                collect_files(val)
        elif isinstance(obj, list):
            for item in obj:
                collect_files(item)

    collect_files(game)
    for filepath, raw_content in files_to_write.items():
        rel_path = filepath.split('source/', 1)[1] if 'source/' in filepath else os.path.basename(filepath)
        full_out_path = os.path.join(output_dir, rel_path)
        os.makedirs(os.path.dirname(full_out_path), exist_ok=True)
        with open(full_out_path, 'w', encoding='utf-8') as f:
            f.write(raw_content)
    
    print(f"SUCCESS: Restored {len(files_to_write)} files to '{output_dir}'.")
    
    # Zip the results
    shutil.make_archive('source_backup', 'zip', output_dir)
    print("Created source_backup.zip")

In [ ]:
URL = "" # @param {type:"string"}
core_js_path = 'core.js'

if URL.strip():
    print(f"Downloading core.js from {URL}...")
    r = requests.get(URL)
    with open(core_js_path, 'wb') as f:
        f.write(r.content)
else:
    print("Please upload core.js file from your computer:")
    uploaded = files.upload()
    if uploaded:
        core_js_path = list(uploaded.keys())[0]

if os.path.exists(core_js_path):
    decompile(core_js_path)
    files.download('source_backup.zip')
else:
    print("No core.js found.")